In [ ]:
# Kundenverwaltung, Preise und Editor-Kapazität

from math import ceil


class PreisEditor:
    def __init__(self, long_preis=0.0, short_preis=0.0,
                 editor_kosten_long=0.0, editor_kosten_short=0.0,
                 kapazitaet=1, fixkosten_monat=0.0):
        self.long_preis = float(long_preis)
        self.short_preis = float(short_preis)
        self.editor_kosten_long = float(editor_kosten_long)
        self.editor_kosten_short = float(editor_kosten_short)
        self.kapazitaet = int(kapazitaet)
        self.fixkosten_monat = float(fixkosten_monat)

    def set_long_preis(self, preis):
        self.long_preis = float(preis)

    def set_short_preis(self, preis):
        self.short_preis = float(preis)

    def set_editor_kosten_long(self, kosten):
        self.editor_kosten_long = float(kosten)

    def set_editor_kosten_short(self, kosten):
        self.editor_kosten_short = float(kosten)

    def set_kapazitaet(self, kapazitaet):
        self.kapazitaet = int(kapazitaet)

    def set_fixkosten_monat(self, fixkosten):
        self.fixkosten_monat = float(fixkosten)

    def umsatz_pro_kunde(self, long_menge=0, short_menge=0):
        return (long_menge * self.long_preis) + (short_menge * self.short_preis)

    def editor_kosten_fuer_videos(self, long_menge=0, short_menge=0):
        return (long_menge * self.editor_kosten_long) + (short_menge * self.editor_kosten_short)

    def marge_pro_long(self):
        return self.long_preis - self.editor_kosten_long

    def marge_pro_short(self):
        return self.short_preis - self.editor_kosten_short

    def __repr__(self):
        return (
            f"PreisEditor(long_preis={self.long_preis}, short_preis={self.short_preis}, "
            f"editor_kosten_long={self.editor_kosten_long}, "
            f"editor_kosten_short={self.editor_kosten_short}, "
            f"kapazitaet={self.kapazitaet}, fixkosten_monat={self.fixkosten_monat})"
        )


class Kunde:
    def __init__(self, name, long_menge=0, short_menge=0):
        self.name = name
        self.long_menge = float(long_menge)
        self.short_menge = float(short_menge)

    def arbeitsmenge_pro_monat(self):
        return self.long_menge + self.short_menge

    def umsatz_pro_monat(self, preis_editor):
        return preis_editor.umsatz_pro_kunde(self.long_menge, self.short_menge)

    def __repr__(self):
        return (
            f"Kunde(name={self.name!r}, long_menge={self.long_menge}, "
            f"short_menge={self.short_menge})"
        )


class KundenVerwaltung:
    def __init__(self, preis_editor):
        self.preis_editor = preis_editor
        self.kunden = []

    def kunde_hinzufuegen(self, kunde):
        self.kunden.append(kunde)

    def long_pro_monat(self):
        return sum(kunde.long_menge for kunde in self.kunden)

    def short_pro_monat(self):
        return sum(kunde.short_menge for kunde in self.kunden)

    def arbeitsmenge_pro_monat(self):
        return sum(kunde.arbeitsmenge_pro_monat() for kunde in self.kunden)

    def umsatz_pro_monat(self):
        return sum(kunde.umsatz_pro_monat(self.preis_editor) for kunde in self.kunden)

    def umsatz_nach_zeit(self, monate):
        return self.umsatz_pro_monat() * int(monate)

    def arbeitsmenge_nach_zeit(self, monate):
        return self.arbeitsmenge_pro_monat() * int(monate)

    def editoren_benoetigt(self, monate=None):
        arbeitsmenge = self.arbeitsmenge_pro_monat()
        if arbeitsmenge == 0:
            return 0
        return ceil(arbeitsmenge / self.preis_editor.kapazitaet)

    def editor_kosten_pro_monat(self):
        return self.preis_editor.editor_kosten_fuer_videos(
            self.long_pro_monat(), self.short_pro_monat()
        )

    def kosten_pro_monat(self):
        return self.editor_kosten_pro_monat() + self.preis_editor.fixkosten_monat

    def kosten_nach_zeit(self, monate):
        return self.kosten_pro_monat() * int(monate)

    def gewinn_pro_monat(self):
        return self.umsatz_pro_monat() - self.kosten_pro_monat()

    def gewinn_nach_zeit(self, monate):
        return self.umsatz_nach_zeit(monate) - self.kosten_nach_zeit(monate)

    def durchschnitts_kunde(self):
        if not self.kunden:
            return None
        long_summe = sum(kunde.long_menge for kunde in self.kunden)
        short_summe = sum(kunde.short_menge for kunde in self.kunden)
        anzahl = len(self.kunden)
        return Kunde("Durchschnitt", long_summe / anzahl, short_summe / anzahl)

    def _leere_zielplanung(self):
        return {
            "kunden": 0,
            "editoren_pro_monat": 0,
            "videos_pro_monat": 0.0,
            "videos_gesamt": 0.0,
            "umsatz_pro_monat": 0.0,
            "umsatz_gesamt": 0.0,
            "kosten_pro_monat": 0.0,
            "kosten_gesamt": 0.0,
            "gewinn_pro_monat": 0.0,
            "gewinn_gesamt": 0.0,
            "marge_prozent": 0.0,
            "marge_pro_long": self.preis_editor.marge_pro_long(),
            "marge_pro_short": self.preis_editor.marge_pro_short(),
        }

    def zielplanung(self, zielgewinn, monate, beispiel_kunde=None):
        zielgewinn = float(zielgewinn)
        monate = int(monate)
        if beispiel_kunde is None:
            beispiel_kunde = self.durchschnitts_kunde()
        if beispiel_kunde is None:
            return self._leere_zielplanung()

        long_kunde = beispiel_kunde.long_menge
        short_kunde = beispiel_kunde.short_menge
        umsatz_kunde_monat = beispiel_kunde.umsatz_pro_monat(self.preis_editor)
        arbeit_kunde_monat = beispiel_kunde.arbeitsmenge_pro_monat()

        if umsatz_kunde_monat <= 0 or arbeit_kunde_monat <= 0:
            return self._leere_zielplanung()

        kunden = 1
        while True:
            long_monat = long_kunde * kunden
            short_monat = short_kunde * kunden
            videos_monat = arbeit_kunde_monat * kunden
            editoren = ceil(videos_monat / self.preis_editor.kapazitaet)
            umsatz_monat = umsatz_kunde_monat * kunden
            editor_kosten_monat = self.preis_editor.editor_kosten_fuer_videos(
                long_monat, short_monat
            )
            kosten_monat = editor_kosten_monat + self.preis_editor.fixkosten_monat
            gewinn_monat = umsatz_monat - kosten_monat
            gewinn_gesamt = gewinn_monat * monate
            if gewinn_gesamt >= zielgewinn:
                marge_prozent = (gewinn_monat / umsatz_monat * 100.0) if umsatz_monat > 0 else 0.0
                return {
                    "kunden": kunden,
                    "editoren_pro_monat": editoren,
                    "videos_pro_monat": videos_monat,
                    "videos_gesamt": videos_monat * monate,
                    "umsatz_pro_monat": umsatz_monat,
                    "umsatz_gesamt": umsatz_monat * monate,
                    "kosten_pro_monat": kosten_monat,
                    "kosten_gesamt": kosten_monat * monate,
                    "gewinn_pro_monat": gewinn_monat,
                    "gewinn_gesamt": gewinn_gesamt,
                    "marge_prozent": marge_prozent,
                    "marge_pro_long": self.preis_editor.marge_pro_long(),
                    "marge_pro_short": self.preis_editor.marge_pro_short(),
                }
            kunden += 1

    def kundenliste(self):
        return list(self.kunden)


preis_editor = PreisEditor(
    long_preis=800,
    short_preis=250,
    editor_kosten_long=200,
    editor_kosten_short=30,
    kapazitaet=5,
    fixkosten_monat=500,
)
verwaltung = KundenVerwaltung(preis_editor)

verwaltung.kunde_hinzufuegen(Kunde("Anna", long_menge=4, short_menge=3))
verwaltung.kunde_hinzufuegen(Kunde("Ben", long_menge=3, short_menge=4))

ergebnis = verwaltung.zielplanung(10000, 3)

print(f"Marge pro Long-Video:  {preis_editor.marge_pro_long():.2f} USD")
print(f"Marge pro Short-Video: {preis_editor.marge_pro_short():.2f} USD")
print()
print(f"Kunden:             {ergebnis['kunden']}")
print(f"Editoren pro Monat: {ergebnis['editoren_pro_monat']}")
print(f"Videos pro Monat:   {ergebnis['videos_pro_monat']}")
print(f"Umsatz pro Monat:   {ergebnis['umsatz_pro_monat']:.2f} USD")
print(f"Kosten pro Monat:   {ergebnis['kosten_pro_monat']:.2f} USD")
print(f"Gewinn pro Monat:   {ergebnis['gewinn_pro_monat']:.2f} USD")
print(f"Marge:              {ergebnis['marge_prozent']:.1f} %")
print(f"Gewinn gesamt (3M): {ergebnis['gewinn_gesamt']:.2f} USD")

ergebnis